<a href="https://colab.research.google.com/github/TechByAniket/Telecom-Customer-Churn-Prediction/blob/main/ADS_Project_Telco_Churn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os

dataset_path = "/content/drive/MyDrive/ADSProject/IBM_Dataset"

print(os.listdir(dataset_path))

['Population.xlsx', 'Location.xlsx', 'Services.xlsx', 'Demographics.xlsx', 'Status.xlsx']


Reading Datasets

In [4]:
import pandas as pd

# Demographics Dataset
demographics_df = pd.read_excel(
    "/content/drive/MyDrive/ADSProject/IBM_Dataset/Demographics.xlsx"
)

# Location Dataset
location_df = pd.read_excel(
    "/content/drive/MyDrive/ADSProject/IBM_Dataset/Location.xlsx"
)

# Population Dataset
population_df = pd.read_excel(
    "/content/drive/MyDrive/ADSProject/IBM_Dataset/Population.xlsx"
)

# Services Dataset
services_df = pd.read_excel(
    "/content/drive/MyDrive/ADSProject/IBM_Dataset/Services.xlsx"
)

# Statud Dataset
status_df = pd.read_excel(
    "/content/drive/MyDrive/ADSProject/IBM_Dataset/Status.xlsx"
)

In [5]:
# Row count of each dataset

print("Demographics:", demographics_df.shape)
print("Services:", services_df.shape)
print("Status:", status_df.shape)
print("Location:", location_df.shape)
print("Population:", population_df.shape)

# Unique CustomerIDs
print("\nNumber of Unique Customer IDs")
print(demographics_df["Customer ID"].nunique())
print(services_df["Customer ID"].nunique())
print(status_df["Customer ID"].nunique())
print(location_df["Customer ID"].nunique())
# print(population_df["Customer ID"].nunique())

Demographics: (7043, 9)
Services: (7043, 31)
Status: (7043, 11)
Location: (7043, 10)
Population: (1671, 3)

Number of Unique Customer IDs
7043
7043
7043
7043


Merge Datasets

In [6]:
master_df = demographics_df.merge(
    services_df,
    on="Customer ID",
    how="inner"
)

master_df = master_df.merge(
    status_df,
    on="Customer ID",
    how="inner"
)

# Same column exists in master_df and location_df, thus error
# master_df = master_df.merge(
#     location_df,
#     on="Customer ID",
#     how="inner"
# )

In [7]:
# Columns of both datasets

print("Location columns:")
print(location_df.columns.tolist())

print("\nMaster columns:")
print(master_df.columns.tolist())

Location columns:
['Location ID', 'Customer ID', 'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude']

Master columns:
['Customer ID', 'Count_x', 'Gender', 'Age', 'Under 30', 'Senior Citizen', 'Married', 'Dependents', 'Number of Dependents', 'Service ID', 'Count_y', 'Quarter_x', 'Referred a Friend', 'Number of Referrals', 'Tenure in Months', 'Offer', 'Phone Service', 'Avg Monthly Long Distance Charges', 'Multiple Lines', 'Internet Service', 'Internet Type', 'Avg Monthly GB Download', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charge', 'Total Charges', 'Total Refunds', 'Total Extra Data Charges', 'Total Long Distance Charges', 'Total Revenue', 'Status ID', 'Count', 'Quarter_y', 'Satisfaction Score', 'Customer Status', 'Churn Label', 'Churn Value', 'Churn Score', 'CLTV', 'Churn Reas

In [8]:
# Check what value the 'Count' column helds

print(master_df["Count"].value_counts())
print(location_df["Count"].value_counts())

Count
1    7043
Name: count, dtype: int64
Count
1    7043
Name: count, dtype: int64


In [9]:
# Removing 'Count' columns, as it doesnt contribute to prediction

master_df = master_df.drop(columns=["Count", "Count_x", "Count_y"])
location_df = location_df.drop(columns=["Count"])

In [10]:
# Checking if there are still any other common columns are present or not

common_cols = set(master_df.columns).intersection(location_df.columns)
print(common_cols)

{'Customer ID'}


In [11]:
# Since no common columns other than 'Customer ID', proceeding to merge
master_df = master_df.merge(
    location_df,
    on="Customer ID",
    how="inner"
)

In [12]:
print(master_df.shape)

(7043, 54)


Merging 'Population' Dataset

In [13]:
print(location_df.columns.tolist())
print(population_df.columns.tolist())

['Location ID', 'Customer ID', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude']
['ID', 'Zip Code', 'Population']


In [14]:
# Merge 'Population' Dataset
# Why left join instead of inner?

# master_df has 7043 customers.
# population_df has 1671 ZIP codes.

# Many customers share the same ZIP code, so 1671 ZIP codes are enough to cover 7043 customers.

# Using a left join ensures:

# Every customer stays in the dataset.
# Each customer gets the population of their ZIP code.
# If a ZIP code is missing from population_df, only the Population value becomes NaN; the customer is not removed.

master_df = master_df.merge(
    population_df,
    on="Zip Code",
    how="left"
)

In [15]:
print(master_df.shape)
print(master_df["Population"].isnull().sum())

(7043, 56)
0


##**Data Cleaning & Preprocessing**


Understanding Data

In [16]:
print("Dataset Shape\n")
print(master_df.shape)

print("\n Dataset Information\n")
master_df.info()

# print("\n Head\n")
# master_df.head()

Dataset Shape

(7043, 56)

 Dataset Information

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 56 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   object 
 1   Gender                             7043 non-null   object 
 2   Age                                7043 non-null   int64  
 3   Under 30                           7043 non-null   object 
 4   Senior Citizen                     7043 non-null   object 
 5   Married                            7043 non-null   object 
 6   Dependents                         7043 non-null   object 
 7   Number of Dependents               7043 non-null   int64  
 8   Service ID                         7043 non-null   object 
 9   Quarter_x                          7043 non-null   object 
 10  Referred a Friend                  7043 non-null   object 
 11  Number 

*   7043 rows × 56 columns
*   No missing values in most columns.
*   Numeric columns have correct data types.

**Things to address**

1. Offer → 3166 non-null (3877 missing)
2. Internet Type → 5517 non-null (1526 missing)
3. Churn Reason → 1869 non-null (5174 missing)
4. Quarter_x and Quarter_y (probably duplicate information)
5. Several ID columns (Customer ID, Service ID, Status ID, Location ID, ID) that likely won't be used for model training.

**Check for duplicates**

In [17]:
duplicate_rows = master_df.duplicated().sum()
print("Duplicate Rows:", duplicate_rows)

Duplicate Rows: 0


**Count and calculate percentage of missing values**

In [18]:
missing = pd.DataFrame({
    "Missing Values": master_df.isnull().sum(),
    "Percentage": (master_df.isnull().sum() / len(master_df)) * 100
})

missing = missing[missing["Missing Values"] > 0]
missing.sort_values(by="Missing Values", ascending=False)

,Missing Values,Percentage
Churn Reason,5174,73.463013
Offer,3877,55.047565
Internet Type,1526,21.666903


In [19]:
master_df[
    master_df["Internet Type"].isnull()
]["Internet Service"].value_counts()

,count
Internet Service,
No,1526


In [20]:
!pip install dvc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.1/470.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.3/79.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.2/451.2 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.2/214.2 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 381.2/38

In [21]:
!git --version
!dvc --version

git version 2.34.1
3.67.1


In [22]:
!git clone https://github.com/TechByAniket/Telecom-Customer-Churn-Prediction.git

Cloning into 'Telecom-Customer-Churn-Prediction'...
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 6 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (6/6), 5.19 KiB | 5.19 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [25]:
%cd Telecom-Customer-Churn-Prediction
!dvc init

/content/Telecom-Customer-Churn-Prediction
Initialized DVC repository.

You can now commit the changes to git.

+---------------------------------------------------------------------+
|                                                                     |
|        DVC has enabled anonymous aggregate usage analytics.         |
|     Read the analytics documentation (and how to opt-out) here:     |
|             <https://dvc.org/doc/user-guide/analytics>              |
|                                                                     |
+---------------------------------------------------------------------+

What's next?
------------
- Check out the documentation: <https://dvc.org/doc>
- Get help and share ideas: <https://dvc.org/chat>
- Star us on GitHub: <https://github.com/treeverse/dvc>
